B"H
# Mitigating an AI Customer Service Agent's Failure to Escalate Requests

Customer service is one of the most in-demand and popular use-cases for Agentic Artificial Intelligence (AI). AI agents are able to resolve somewhere between 75% of user queries, and that figure is expected to rise to 95% by the end of 2025[^1]. There remain cases, however, where escalation to a human representative is still desired. For example, if the user expresses repeated frustration and threatens legal action, it might be in the organization's best interest to escalate the ticket to a human representative.

Here, I aim for a solution which mitigates a real bug encountered by a leading chat AI company. In certain scenarios which warranted escalation, the AI agent failed to call the correct tool; it merely responded with a message indicating that the ticket was being transferred to a human, without any attempt to call the relevant tool for escalating.

With technical factors (eg. context limit) ruled out, the most plausible explanation for the bug was "hallucinatory", eg. all of the correct factors (tool definition, system prompt, normal escalation conditions) were in-place, but the LLM sometimes just failed to take the correct action. This category of bug is all-to-familiar for anyone who's worked with integrating LLMs and in this notebook, I aim to provide a solution.

[1] https://masterofcode.com/blog/ai-in-customer-service-statistics

## Assumptions

I'm assuming the following about the failed escalations related to this bug:
- The LLM's context (maximum token count) was not met. Similarly, other technical factors, such as tool definition, were all correct.
- Escalation conditions _were_ met (eg. the user sent extremely angry messages and/or legal threats).
- The LLM output a message saying that it has transferred (or will transfer) the user to a human representative, but no action was taken.

And these are my assumptions for the patch:
- Our code has access to the user's message pre-inference.
- Our code has access to the assistant's response.
- One message suffices to decide that the chat should be escalated - eg. larger context (more than one message) is not required for this task.
- We track and have access to the chat's `state`, including if it's been `escalted`.

## My approach

I propose 2 approaches to quickly mitigate this bug in production:
1. Pass each `user` and `assistant` message through a flagged-keyword/phrase filter and (optionally) a low-cost LLM to classify whether the chat should be escalated.
2. Build a lightweight NLP binary classifier to replace the LLM in Step 1.

### The patch at a glance
This patch targets two specific points in the application flow:
1. After a new message is submitted by the user, before it's passed to the LLM.
2. After the assistant's text response is received and parsed, before it is streamed to the UI.

The patch will check:

- A. If the user or assistant message indicates escalation
- B. If the chat has NOT been escalated

If both A and B are `True`, the patch triggers escalation manually. At a high-level, it looks something like this:

```python
# high-level pseudocode for the patch
def facilitate_potential_escalation(message, role, chat_state):
    """ Patch for failed escalation bug. 
        Checks if a user or assistant message indicates escalation and the chat wasn't escalated.
        If so, trigger the escalation manually.
    """
    # Only proceed checks if the chat was never escalated
    if not chat_state.escalated:

        # check if message indicates that we SHOULD escalate
        should_escalate = check_msg_for_escalation(message, role)

        # if we SHOULD escalate, do so
        if should_escalate:
            escalate()
```

And the following are the locations in the application flow for inserting this patch:

1. After the user submits a new message (before submitting it to the chat assistant):

```python
# new message received from user
# ...
facilitate_potential_escalation(message, role='user', chat_state)
# ...
```

2. After an assistant message is parsed:

```python
# new message received from assistant
# ...
facilitate_potential_escalation(message, role='assistant', chat_state)
# ...
```

Downstream business logic varies based on application specifics, but this is enough to patch the bug and _escalate the request if the LLM assistant fails to do so._

## Flagging keywords and phrases for escalation

The first step is to formulate a list of flagged keywords and phrases which either trigger escalation or indicate that the LLM intended to do so.

### From the user

For reference, the following are example **user** messages which warrant escalation:

> "This is unacceptable. If I don’t speak to a real person now, I’ll be contacting my bank and filing a complaint under the Fair Credit Billing Act."

> "If I don’t get a live agent now, I will file a formal complaint with the FTC for breach of delivery terms under the Mail Order Rule."

> "I’m requesting immediate escalation. If I don't speak to a human agent, I’ll be lodging a legal complaint with the Information Commissioner’s Office."

Formulated as a Python Array, I define `USER_FLAGGED_KEYWORDS`:

In [1]:
COMPANY_NAME = 'AIChat'
USER_FLAGGED_KEYWORDS = [
    'complaint',
    'legal',
    'illegal',
    'lawsuit',
    'breach',
    'dispute',
    'lawsuit',
    'sue you',
    'sue yall',
    f'sue {COMPANY_NAME}',
    # optionally add more (eg. curse words) here
]

### From the assistant
From the assistant's side, I'll define `ASSISTANT_FLAGGED_KEYWORDS` to catch the assistant's intent to escalate:

In [2]:
ASSISTANT_FLAGGED_KEYWORDS = [
    'transfer you to',
    'transferring you to',
    'transferred',
    'human rep',
    'human representative',
    'human customer service',
    'escalate',
    'escalated',
    'escalating',
]

## Flag detection pipeline

Now, I need to implement a pipeline for detecting the presence of flagged keywords or phrases in a message.

### Part 1: Message normalization
Messages must first be converted into an array of normalized words:

In [3]:
from typing import List
import re

def split_normalize_message(message: str) -> List[str]:
    """ Convert raw message string to a normalized array """
    
    # normalize message to lower-case and strip special characters
    normalized_msg = re.sub(r'[^a-z\s]', '', message.lower())
    
    # convert message to array of words
    split_msg = normalized_msg.split()
    
    return split_msg

Example output:

In [4]:
message = "HEY,,,  I got the wrong size shirt in my order!!"
split_normalize_message(message)

['hey', 'i', 'got', 'the', 'wrong', 'size', 'shirt', 'in', 'my', 'order']

### Part 2: Escalation-flag detection

Next, I need a function which parses the normalized message array and detects the presence of either a **flagged keyword** or **phrase**:

In [5]:
from typing import Literal

def detect_escalation_flags(
    normalized_msg_array: List[str],
    role: Literal['user', 'assistant']
) -> bool:
    """
    Returns True if a flagged keyword or phrase is detected in a normalized message array, else False
    """

    # Use correct flag definition based on role (user / assistant)
    flags = USER_FLAGGED_KEYWORDS if role=='user' else ASSISTANT_FLAGGED_KEYWORDS

    # re-join normalized message to check for flagged phrases
    joined = ' '.join(normalized_msg_array)

    for keyword in flags:

        # check for presence of flagged keyword
        if keyword in normalized_msg_array:
            return True
        
        # check for presence of flagged phrases (they contain spaces)
        if " " in keyword and keyword in joined:
            return True

    return False # this is reached if no flags are present - no escalation required!

Example output:

In [6]:
print(detect_escalation_flags(['hey', 'i', 'got', 'the', 'wrong', 'size', 'shirt', 'in', 'my', 'order'], 'user'))
print(detect_escalation_flags(['im', 'gonna', 'sue', 'you'], 'user'))
print(detect_escalation_flags(['my', 'name', 'is', 'sue', 'smith'], 'user'))
print(detect_escalation_flags(['i', 'am', 'filing', 'a', 'complaint', 'with'], 'user'))

False
True
False
True


### Part 3: High-level API

Next, I need a function which can process a `user` or `assistant` message through the pipeline and formulate a result (`should_escalate = True/False`):

In [7]:
def check_msg_for_escalation(
    message: str,
    role: Literal['user', 'assistant']
) -> bool:
    """
    Returns True if a user or assistant message should trigger escalation, else False
    """

    # convert message to normalized array of words
    normalized_split_msg = split_normalize_message(message)

    # check if message should trigger escalation
    should_escalate = detect_escalation_flags(normalized_split_msg, role)

    return should_escalate

Finally, I need a function which orchestrates the patch end to end.
Ideally, this funciton can access the chat's current `escalated` state. In the example below, the state is tracked via a thread-safe `threading.Event` object.

In [8]:
from dataclasses import dataclass, field
from threading import Event

@dataclass
class ChatState:
    escalated: Event = field(default_factory=Event) # thread-safe event tracking chat's escalation state

def escalate(chat_state: ChatState):
    """ Function which triggers actual escalation hand-off to human rep """
    chat_state.escalated.set() # update state
    # escalation logic here...
    print('Chat escalated to a human rep')

def facilitate_potential_escalation(
    message: str,
    role: Literal['user', 'assistant'],
    chat_state: ChatState,
):
    """ Patch for failed escalation bug. 
        Checks if a user or assistant message indicates escalation and the chat wasn't escalated.
        If so, trigger the escalation manually.
    """
    # Only proceed checks if the chat was never escalated
    if not chat_state.escalated.is_set():

        # check if message indicates that we SHOULD escalate
        should_escalate = check_msg_for_escalation(message, role)

        # if we SHOULD escalate, do so
        if should_escalate:
            escalate(chat_state)

### Part 4: Adding NLP binary classifiers to the pipeline

To strengthen this patch by making it a lot more robust, I'm going to add an optional step with 2 NLP binary classifiers (for user and assistant messages). In order to deliver the patch as fast as possible, I'm going to start by making the "engine" for this classifier a low-cost LLM. In order to reduce costs, a homebrewed traditional NLP model will be built later to replace the LLM.

#### Entry point

The classifiers will be added to the function which decides if escalation is warranted (`check_msg_for_escalation`), with an optional flag (default `True`) for using them:

In [9]:
# boilerplate for new classification function
def run_nlp_escalation_classifier(
    message: str, # change to List[str] and pass `normalized_split_msg` when replacing the LLM with a traditional NLP model
    role: Literal['user', 'assistant'],
) -> bool:
    """
    Processes a user or assistant message with a binary classifier.
    Returns True to indicate that escalation is warranted, else False.
    """
    return False # boilerplate for now. implementation is coming in following steps.

# update deciding function
def check_msg_for_escalation(
    message: str,
    role: Literal['user', 'assistant'],
    use_classifiers: bool = True,
) -> bool:
    """
    Returns True if a user or assistant message should trigger escalation, else False
    """

    # convert message to normalized array of words
    normalized_split_msg = split_normalize_message(message)

    # check if message should trigger escalation
    should_escalate = detect_escalation_flags(normalized_split_msg, role)

    # run messages through optional classifier, if needed
    if use_classifiers and not should_escalate:
        should_escalate = run_nlp_escalation_classifier(message, role)

    return should_escalate

#### Implementation: Using an LLM with Structured Output

As discussed, the fastest way to deliver this feature is by leveraging an LLM, and a homebrewed replacement for cost savings can be implemented later.
To implement this, I'll initialize an OpenAI API client and write a function which leverages a low-cost model with a system message and Structured Output schema:

##### Initializing the OpenAI API client
This step requires two additional dependencies: `python-dotenv` and `openai`.

An `OPENAI_API_KEY` is defined in the `.env` file (see `.env.example`).

In [10]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
OPENAI_CLIENT = OpenAI()

##### Designing the prompt and Structured Output schema

This is not a complex task, so a low-cost model should suffice. The [cheapest OpenAI model](https://platform.openai.com/docs/pricing) right now which supports the Structured Outputs API is `gpt-4.1-nano`, followed by `gpt-4o-mini`. So, I will start with those.

According to the [official docs](https://platform.openai.com/docs/guides/structured-outputs?api-mode=responses&example=structured-data#introduction), "Structured Outputs is a feature that ensures the model will always generate responses that adhere to your supplied JSON Schema, so you don't need to worry about the model omitting a required key, or hallucinating an invalid enum value." That makes it very easy to interact with the LLM for this feature. All I have to do is define a Pydantic model with a required key for binary classification:

In [11]:
from pydantic import BaseModel

OPENAI_MODEL = 'gpt-4.1-nano-2025-04-14' # can also try 'gpt-4o-mini-2024-07-18'

# prompts for user and assistant message classification
SYS_PROMPT_PARSE_USER_MSG = 'Does the attached customer message imply extreme frustration, intention to cancel service, or threaten legal action?'
SYS_PROMPT_PARSE_ASSISTANT_MSG = 'Does the attached customer service message imply intention to escalate the request to a human representative?'

# response schema
class EscalationVerdict(BaseModel):
    should_escalate: bool

##### Completing the function

In [12]:
def classify_escalation_openai(
    message: str,
    role: Literal['user', 'assistant'],
    openai_client: OpenAI,
    openai_model: str
) -> bool:
    """ Classify a message for escalation with OpenAI Structured Output API """

    # use the correct system prompt based on role
    system_prompt = SYS_PROMPT_PARSE_USER_MSG if role == 'user' else SYS_PROMPT_PARSE_ASSISTANT_MSG

    # invoke OpenAI for a response
    response = openai_client.responses.parse(
        model = openai_model,
        instructions = system_prompt,
        input = [
            {
                "role": "system",
                "content": system_prompt,
            },
            {"role": "user", "content": message},
        ],
        text_format = EscalationVerdict,
    )

    # parse response
    escalation_verdict = response.output_parsed
    
    return escalation_verdict.should_escalate

##### Integration into existing pipeline

Now I can update `run_nlp_escalation_classifier()` with its implementation:

In [13]:
def run_nlp_escalation_classifier(
    message: str, # change to List[str] and pass `normalized_split_msg` when replacing the LLM with a traditional NLP model
    role: Literal['user', 'assistant'],
    openai_client: OpenAI = OPENAI_CLIENT,
    openai_model: str = OPENAI_MODEL
) -> bool:
    """
    Processes a user or assistant message with a binary classifier.
    Returns True to indicate that escalation is warranted, else False.
    """
    
    should_escalate = classify_escalation_openai( # easily swapped with traditional NLP function later
        message,
        role,
        openai_client,
        openai_model
    )
    
    return should_escalate

Here's a quick demo:

In [14]:
tests = [
    {
        'message': "I'm sorry. You are being transferred to a live agent.",
        'role': 'agent',
        'expected_answer': True
    },
    {
        'message': "Thank you. I hope that solved your inquiry!",
        'role': 'agent',
        'expected_answer': False
    },
    {
        'message': "If i don't get transferred immedaitely, i'm fililng a dispute with my bank",
        'role': 'user',
        'expected_answer': True
    },
    {
        'message': "Hey, i need help with my order it never showed up!!",
        'role': 'user',
        'expected_answer': False
    },   
]

for test in tests:
    print("Raw test data:", test, '\n')
    
    result = run_nlp_escalation_classifier(
        message = test["message"],
        role = test["role"],
        openai_client = OPENAI_CLIENT,
        openai_model = OPENAI_MODEL
    )
    print("Result:", result)
    print("Expected:", test["expected_answer"], '\n------------')


Raw test data: {'message': "I'm sorry. You are being transferred to a live agent.", 'role': 'agent', 'expected_answer': True} 

Result: True
Expected: True 
------------
Raw test data: {'message': 'Thank you. I hope that solved your inquiry!', 'role': 'agent', 'expected_answer': False} 

Result: False
Expected: False 
------------
Raw test data: {'message': "If i don't get transferred immedaitely, i'm fililng a dispute with my bank", 'role': 'user', 'expected_answer': True} 

Result: True
Expected: True 
------------
Raw test data: {'message': 'Hey, i need help with my order it never showed up!!', 'role': 'user', 'expected_answer': False} 

Result: False
Expected: False 
------------


### Part 5: Integration and testing

The patch is now ready for integration into the application. As outlined above (see [The patch at a glance](#the-patch-at-a-glance)), every new user and assistant message is submitted to the patch's entry function, `facilitate_potential_escalation()`.

To test the patch, I'll run it against a variety of examples of both user and assistant messages, which should or shouldn't trigger escalation:

In [15]:
# define test cases
assistant_msgs_escalate = [
    'Your request is being transferred.',
    'I\'m sorry. Your inquiry is being elevated to a human representative.',
    'You are being transferred to a human rep',
    'Your request has been escalated to a live representative. Please wait.',
    "I'm sorry. You are being transferred to a live agent."
]
assistant_msgs_clean = [
    'Your refund has been processed successfully. Is there anything else I can assist you with today?',
    'The estimated delivery date for your order is Thursday, July 25 between 6-7pm.',
    'Hi Johnathon, how can I assist you?',
    'Please provide your order number for me to assist you.',
    "Thank you. I hope that solved your inquiry!"
]
user_msgs_escalate = [
    'You guys just took my money for nothing. I\'ll have to file a dispute with my credit card!',
    'I\'m contacting my lawyer immediately. If this isn\'t resolved today, you will be facing a lawsuit.',
    'This is an absolute breach of our agreement. I\'m going to be taking action!',
    'i hate this app i\'m gonna sue y\'all',
    "If i don't get transferred immedaitely, i'm fililng a dispute with my bank",
    'This is absolute garbage. I\'ve been a loyal subscriber for years, guess I\'ll have to finally cancel!'
]
user_msgs_clean = [
    'Hey i have an issue with my order #889854',
    'What is the shipping status?',
    'Do you offer any promos?',
    'Thanks!',
    "Hey, i need help with my order it never showed up!!"
]

Running the tests:

In [16]:
# initiate counter
escalation_count = 0
expected_escalations = len(assistant_msgs_escalate) + len(user_msgs_escalate)

# test escalation flag detection (user)
for msg in user_msgs_escalate:
    state = ChatState()
    facilitate_potential_escalation(msg, 'user', state)
    assert state.escalated.is_set(), f'Failed for message: \'{msg}\'' # ensure escalation
    escalation_count += 1 # increment counter
for msg in user_msgs_clean:
    state = ChatState()
    facilitate_potential_escalation(msg, 'user', state)
    assert not state.escalated.is_set(), f'Failed for message: \'{msg}\'' # ensure no escalation
    
# test escalation flag detection (assistant)
for msg in assistant_msgs_escalate:
    state = ChatState()
    facilitate_potential_escalation(msg, 'assistant', state)
    assert state.escalated.is_set(), f'Failed for message: \'{msg}\'' # ensure escalation
    escalation_count += 1
for msg in assistant_msgs_clean:
    state = ChatState()
    facilitate_potential_escalation(msg, 'assistant', state)
    assert not state.escalated.is_set(), f'Failed for message: \'{msg}\'' # ensure no escalation

print('All tests passed!' if escalation_count == expected_escalations else '') # extra check - this line is only reached if all tests pass anyway

Chat escalated to a human rep
Chat escalated to a human rep
Chat escalated to a human rep
Chat escalated to a human rep
Chat escalated to a human rep
Chat escalated to a human rep
Chat escalated to a human rep
Chat escalated to a human rep
Chat escalated to a human rep
Chat escalated to a human rep
Chat escalated to a human rep
All tests passed!


## Conclusion

In this notebook, I've addressed a critical production defect faced by AI customer service agents, where the agent claims to escalate high-risk conversations but fails to actually do so. I implemented a lightweight flag‑detection safeguard, which is simple to insert into the existing workflow. This patch helps restore reliable, audit‑ready escalation without real modifications to the core agent code or infrastructure. The result is higher customer satisfaction, lower risk, and an extensible path forward for further optimisations.

### Highlights

 - Every user and assistant message is screened for escalation cues, and escalation is triggered only if the AI agent failed to do so.
 - This patch employs a simple flagged keywords/phrases filter as a first layer of defense, with an optional LLM/NLP binary classifier that catches subtler wording.
 - The patch and all of its code has been successfully validated with unit tests.

### Next steps

- Build and evaluate homebrewed NLP models to replace the LLM in the binary classification step.
- Expand the test library with more test cases, including snippets from real production data.
- Extend the pipeline to collect multiple user messages at a time to catch more nuanced escalation cues.